# Setup and dataset

In [1]:
import torch
import torchvision.transforms as T

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
import sys
sys.path.append('..')
from data_processing.data_utils import get_domain_dataloaders_from_hdf5

domains = ['Home', 'BigOffice-2', 'BigOffice-3', 'Hallway', 'MeetingRoom', 'SmallOffice']
cols = ['image_path']

# DARE

Run with CLI command:

python main.py --experiment_id officemanners_dil_fold0 --model maxd_ema --dataset custom-hdf5-regression --img_size 64 --num_tasks 6 --alpha 0.1 --beta 0.2 --maximize_task hcr --maxd_weight 0.1 --mind_weight 1 --logitb_weight 1 --logitc_weight 1 --iterative_buffer --supcon_weight 0.0 --supcon_temp 0.8 --frozen_supcon --intermediate_sampling --std 4 --reduce_lr --each_epoch --buffer_size 120 --lr 0.04 --batch_size 16 --minibatch_size 16 --n_epochs 50 --output_folder ./output --tensorboard --plot_results

in .bat

```
for /L %i in (0,1,4) do (
    python main.py --experiment_id officemanners_dil_fold%%i --model maxd_ema --dataset custom-hdf5-regression --img_size 64 --num_tasks 6 --alpha 0.1 --beta 0.2 --maximize_task hcr --maxd_weight 0.1 --mind_weight 1 --logitb_weight 1 --logitc_weight 1 --iterative_buffer --supcon_weight 0.0 --supcon_temp 0.8 --frozen_supcon --intermediate_sampling --std 4 --reduce_lr --each_epoch --buffer_size 120 --lr 0.04 --batch_size 16 --minibatch_size 16 --n_epochs 50 --output_folder ./output --tensorboard --plot_results
)
```


In [3]:
def infer_per_domain(model, device, domain_dataloaders, eval_subset):
    domain_data = {}
    model.eval()
    with torch.no_grad():
        for domain, loaders in domain_dataloaders.items():
            domain_preds = []
            domain_labels = []
            for batch in loaders[eval_subset]:
                inputs, labels, _ = batch
                inputs = tuple(x.to(device) for x in inputs)
                labels = labels.to(device)

                outputs = model(*inputs)['logits1']

                domain_preds.append(outputs.detach())
                domain_labels.append(labels.detach())

            domain_preds = torch.vstack(domain_preds).cpu()
            domain_labels = torch.vstack(domain_labels).cpu()
            
            domain_data[domain] = (domain_preds, domain_labels)
    
    return domain_data

In [4]:
import sys
sys.path.append(r"D:\projects\DARE")
from backbone.ResNet18 import resnet18

model = resnet18(9, norm_feature=False, diff_classifier=False, num_rot=0, ema_classifier=False, 
         lln=False, algorithm='maxd_ema', pretrained=False)

model = model.to(DEVICE)

In [6]:
from pathlib import Path

fold_paths = {
    0:'officemanners_b60_dil_fold0_20260227_100503_182830',
    1:'officemanners_b60_dil_fold1_20260227_114703_586849',
    2:'officemanners_b60_dil_fold2_20260227_133118_231570',
    3:'officemanners_b60_dil_fold3_20260227_153106_197467',
    4:'officemanners_b60_dil_fold4_20260227_171358_574253',
}

model_results = {}
for fold_num, path in fold_paths.items():
    checkpoint_path = f"D:/projects/DARE/output/tensorboard_runs/domain-2il/custom-hdf5-regression/maxd_ema/buf_60/{path}/checkpoint_6.pth"
    checkpoint_weights = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(checkpoint_weights, strict=True)

    # transforms = [T.Compose([T.Resize((90,160))])]
    dataloader = get_domain_dataloaders_from_hdf5(hdf5_path=f"../data/mean_data_pepper_fold{fold_num}.hdf5", domains=domains, img_path_cols=cols)#, transforms=transforms)
    domain_test_preds_targets = infer_per_domain(model, DEVICE, dataloader, 'test')
    
    model_results[f'dare_90160_b60_fold={fold_num}'] = {"domain_test_preds_targets": domain_test_preds_targets}

In [7]:
import pickle
with open('./results/dare_90160_b60_domain_preds_targets_folds.pkl', 'wb') as f:
    pickle.dump(model_results, f)

# DUCA

Trained with CLI command:

python main.py --experiment_id officemanners_dil_fold0 --seed 42 --model duca --dataset custom-hdf5-regression --buffer_size 120 --aux shape --lr 0.05 --n_epochs 50 --img_size 64 --shape_filter sobel --batch_size 8 --minibatch_size 8 --output_dir ./output/ --loss_wt 0.1 0.1 0.01 0.01 --ema_alpha 0.999 --ema_update_freq 0.06 --loss_type l2 --save_model --alpha_mm 0.1 0.1 --beta_mm 0.1 0.1

in .bat

```
for /L %i in (0,1,4) do (
    python main.py --experiment_id officemanners_dil_fold%%i --seed 42 --model duca --dataset custom-hdf5-regression --buffer_size 120 --aux shape --lr 0.05 --n_epochs 50 --img_size 64 --shape_filter sobel --batch_size 8 --minibatch_size 8 --output_dir ./output/ --loss_wt 0.1 0.1 0.01 0.01 --ema_alpha 0.999 --ema_update_freq 0.06 --loss_type l2 --save_model --alpha_mm 0.1 0.1 --beta_mm 0.1 0.1
)
```

In [3]:
def infer_per_domain(model, device, domain_dataloaders, eval_subset):
    domain_data = {}
    model.eval()
    with torch.no_grad():
        for domain, loaders in domain_dataloaders.items():
            domain_preds = []
            domain_labels = []
            for batch in loaders[eval_subset]:
                inputs, labels, _ = batch
                inputs = tuple(x.to(device) for x in inputs)
                labels = labels.to(device)

                outputs = model(*inputs)

                domain_preds.append(outputs.detach())
                domain_labels.append(labels.detach())

            domain_preds = torch.vstack(domain_preds).cpu()
            domain_labels = torch.vstack(domain_labels).cpu()
            
            domain_data[domain] = (domain_preds, domain_labels)
    
    return domain_data

In [4]:
import sys
sys.path.append(r"D:\projects\DUCA")

In [7]:
model_results = {}
for fold_num in range(5):
    checkpoint_path = f'D:/projects/DUCA/output/results/class-il/custom-hdf5-regression/duca/officemanners_b60_dil_fold{fold_num}/net_5.pth'
    model = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model = model.to(DEVICE)

    # transforms = [T.Compose([T.Resize((90,160))])]
    dataloader = get_domain_dataloaders_from_hdf5(hdf5_path=f"../data/mean_data_pepper_fold{fold_num}.hdf5", domains=domains, img_path_cols=cols)#, transforms=transforms)
    domain_test_preds_targets = infer_per_domain(model, DEVICE, dataloader, 'test')

    model_results[f'duca_90160_b60_fold={fold_num}'] = {"domain_test_preds_targets": domain_test_preds_targets}

In [8]:
import pickle
with open('./results/duca_90160_b60_domain_preds_targets_folds.pkl', 'wb') as f:
    pickle.dump(model_results, f)